### Data Ingestion

In [66]:
## Document data structure 

from langchain_core.documents import Document

In [67]:
doc = Document (
    page_content="This is the main text content and I am using it to create a RAG", # This will have the content of the file
    # We will use the metadata in the searching in the vector database
    metadata={
        "Source": "example.txt",
        "pages": 1,
        "Author": "Yazan",
        "Date_created": "2026-6-16"
    }
)

In [68]:
## Create a txt file

import os
os.makedirs("../data/text_files", exist_ok=True)

In [69]:
sample_texts={
    "../data/text_files/java.txt":"""Java Programming Language 

Java is a high-level, object-oriented programming language originally developed by Sun Microsystems and now maintained by Oracle. It was designed with the principle of “write once, run anywhere,” meaning that compiled Java code can run on any device that supports the Java Virtual Machine (JVM) without needing to be recompiled. This makes Java highly portable across different operating systems such as Windows, macOS, and Linux.

Java is widely used in enterprise applications, web backends, Android mobile development, and large-scale systems. Its syntax is similar to C++, but it removes many low-level complexities like manual memory management by using automatic garbage collection. This helps reduce memory leaks and makes development safer and more stable.""",

"../data/text_files/machine_learning.txt": """Machine Learning basics 

Machine Learning is a branch of artificial intelligence that focuses on building systems that can learn from data instead of being explicitly programmed. Instead of writing strict rules for every situation, you provide the machine with examples, and it learns patterns from them. Over time, it can make predictions or decisions based on new, unseen data. This is similar to how humans learn from experience, but in ML the “experience” comes from data.

There are several main types of machine learning. In supervised learning, the model is trained on labeled data, meaning each input has a correct answer. For example, you might train a model to detect spam emails using examples of emails labeled as “spam” or “not spam.” In unsupervised learning, the data has no labels, and the model tries to find hidden patterns or groupings, such as clustering customers based on their buying behavior. A third type, reinforcement learning, involves an agent learning by interacting with an environment and receiving rewards or penalties, like training a model to play a game.

Machine learning is widely used in real-world applications. It powers recommendation systems on platforms like YouTube and Netflix, helps detect fraud in banking, enables voice assistants like Siri and Alexa, and is used in medical diagnosis systems. Even simple things like predicting house prices or filtering emails rely on ML techniques. Because data is growing rapidly, ML has become essential for turning that data into useful insights.

At its core, machine learning depends on three key components: data, features, and models. Data is the raw information collected from the real world. Features are the important parts of that data that the model uses for learning. The model is the algorithm that finds patterns in the features and makes predictions. The quality of the data and features often has a bigger impact on performance than the complexity of the model itself.

Overall, machine learning is powerful because it allows computers to improve automatically with experience. However, it also has challenges, such as needing large amounts of data, being sensitive to poor-quality data, and sometimes producing biased or incorrect results. Despite these challenges, ML continues to grow and is becoming a core technology in fields like healthcare, finance, robotics, and natural language processing.


"""
}

# sample_texts is a dictionary.
# .items() gives you pairs of:
# filepath → the file name/path (like "file1.txt")
# content → the text you want to write inside that file 

for filepath, content in sample_texts.items():
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)

print("Sample text files created")

Sample text files created


In [70]:
## Textloader (to read the content of a file using langchain)

from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/java.txt", encoding="utf-8")
document = loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/java.txt'}, page_content='Java Programming Language \n\nJava is a high-level, object-oriented programming language originally developed by Sun Microsystems and now maintained by Oracle. It was designed with the principle of “write once, run anywhere,” meaning that compiled Java code can run on any device that supports the Java Virtual Machine (JVM) without needing to be recompiled. This makes Java highly portable across different operating systems such as Windows, macOS, and Linux.\n\nJava is widely used in enterprise applications, web backends, Android mobile development, and large-scale systems. Its syntax is similar to C++, but it removes many low-level complexities like manual memory management by using automatic garbage collection. This helps reduce memory leaks and makes development safer and more stable.')]


In [71]:
### Directory loader

from langchain_community.document_loaders import DirectoryLoader

# Load all the text files from the directory 
dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", # Pattern to match files 
    loader_cls= TextLoader, 
    loader_kwargs={'encoding': 'utf-8'},
    show_progress= False

)

documents=dir_loader.load()

documents

[Document(metadata={'source': '..\\data\\text_files\\java.txt'}, page_content='Java Programming Language \n\nJava is a high-level, object-oriented programming language originally developed by Sun Microsystems and now maintained by Oracle. It was designed with the principle of “write once, run anywhere,” meaning that compiled Java code can run on any device that supports the Java Virtual Machine (JVM) without needing to be recompiled. This makes Java highly portable across different operating systems such as Windows, macOS, and Linux.\n\nJava is widely used in enterprise applications, web backends, Android mobile development, and large-scale systems. Its syntax is similar to C++, but it removes many low-level complexities like manual memory management by using automatic garbage collection. This helps reduce memory leaks and makes development safer and more stable.'),
 Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning basics \n\nMa

In [72]:

## Loading a PDF 

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls= PyMuPDFLoader,
    show_progress=False
)


pdf_documents = dir_loader.load()
pdf_documents


[Document(metadata={'producer': 'macOS Version 15.7.4 (Build 24G517) Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20260309213150Z00'00'", 'source': '..\\data\\pdf\\How to Pivot into an Ai_ML Engineering Role in 2026.pdf', 'file_path': '..\\data\\pdf\\How to Pivot into an Ai_ML Engineering Role in 2026.pdf', 'total_pages': 22, 'format': 'PDF 1.4', 'title': 'How to Pivot into an Ai:ML Engineering Role in 2026', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20260309213150Z00'00'", 'trapped': '', 'modDate': "D:20260309213150Z00'00'", 'creationDate': "D:20260309213150Z00'00'", 'page': 0}, page_content='HOW TO PIVOT INTO AN \nAI/ML Engineering Role \nIN 2026 \nThe Step-by-Step Field Guide for Software Engineers \nWith 2–6 Years of Experience \nIncludes Free Learning Resources  •  Job Market Data  •  Proven Roadmap \nUpdated March 2026'),
 Document(metadata={'producer': 'macOS Version 15.7.4 (Build 24G517) Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:2

## Embedding and VectoreDB

In [73]:
import numpy as np
from sentence_transformers import SentenceTransformer # The ebmedding model will be here 
import chromadb
from chromadb.config import Settings 
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [74]:
class EmbeddingManager:

    def __init__(self, model_name: str = 'all-MiniLM-L6-V2'): # Model_name: HuggingFace model name for sentence embeddings
                                        # all-MiniLM-L6-V2 is responsible for converting text to vectors 

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        # Load the SentenceTransformer model

        try:
            print(f"Loading embedding model {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise


    def generate_embeddings(self, texts: List[str]) -> np.ndarray: # It will return a numpy array

        # Geneerate embeddings for a list of texts 

        # texts: List of text strings to embed 

        # Returns: numpy array of embeddings with shape (len(texts), embeddings_dim)

        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generating embeddings with shape: {embeddings.shape}")
        return embeddings
    

## initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model all-MiniLM-L6-V2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8316.90it/s]


Model Loaded successfully. Embedding dimension: 384


C:\Users\yazan\AppData\Local\Temp\ipykernel_35404\2535007295.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model Loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [75]:
class VectorStore:

    # Manages document embeddings in a ChromaDB vector store

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
                                                                # presist_directory means save the vector_store here 
        
        # Initialize the vector store

        # collection_name: Name of the ChromaDB collection 
        # persist_directory: Directory to persist the vector store

        self.collection_name = collection_name
        self.persist_directory =  persist_directory

        self.client = None
        self.collection = None
        self._initialize_store()


    def _initialize_store(self):
    
        # Initialize ChromaDB client and collection

        try:

            # Create persistent ChromaDB client
            
            os.makedirs(self.persist_directory, exist_ok=True)
            # Create a client, which will be having a reference to the ChromaDB vector store
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create a collection 
            # Where will the vectors will be stored
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection : {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")


        except Exception as e:

            print(f"Error initizlizing vector store {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID for a specific record in the vectorDB 
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vector_store = VectorStore()
vector_store

Vector store initialized. Collection : pdf_documents
Existing documents in collection: 48


In [76]:
pdf_documents

[Document(metadata={'producer': 'macOS Version 15.7.4 (Build 24G517) Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20260309213150Z00'00'", 'source': '..\\data\\pdf\\How to Pivot into an Ai_ML Engineering Role in 2026.pdf', 'file_path': '..\\data\\pdf\\How to Pivot into an Ai_ML Engineering Role in 2026.pdf', 'total_pages': 22, 'format': 'PDF 1.4', 'title': 'How to Pivot into an Ai:ML Engineering Role in 2026', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20260309213150Z00'00'", 'trapped': '', 'modDate': "D:20260309213150Z00'00'", 'creationDate': "D:20260309213150Z00'00'", 'page': 0}, page_content='HOW TO PIVOT INTO AN \nAI/ML Engineering Role \nIN 2026 \nThe Step-by-Step Field Guide for Software Engineers \nWith 2–6 Years of Experience \nIncludes Free Learning Resources  •  Job Market Data  •  Proven Roadmap \nUpdated March 2026'),
 Document(metadata={'producer': 'macOS Version 15.7.4 (Build 24G517) Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:2

In [77]:
# We will extract all the text from a particular document or cheunk and generate an embedding 

texts = [doc.page_content for doc in pdf_documents]
texts

# Generate the embeddings 

embeddings = embedding_manager.generate_embeddings(texts)

# Stor in the vector database

vector_store.add_documents(pdf_documents, embeddings)

Generating embeddings for 24 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.51it/s]

Generating embeddings with shape: (24, 384)
Adding 24 documents to vector store...
Successfully added 24 documents to vector store
Total documents in collection: 72


## Retriever Pipeline From VectorStore

In [78]:
# We will convert the query of the user to an embedding

class RAGRetriever:

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        
        # Initialize the retriever

        # vector_store: Vectore store containing document embeddings 
        # embedding_manager: Manager for generating query embeddings

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vector_store,embedding_manager)

In [79]:
rag_retriever

In [82]:
rag_retriever.retrieve("machine learning")

Retrieving documents for query: 'machine learning'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 120.98it/s]

Generating embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]